# YeastCaduceus — Phase 3: Supervised Fine-tuning (coverage prediction)

Predict RNA-seq/ChIP **coverage** (512 bins × 16 bp per 8,192 bp window) for the
4,201 BigWig tracks, on the **R64** genome, from the **domain-adapted** PlantCAD2
backbone produced in Phase 2.

**Design (pinned in `dir/YeastCaduceus_WBS.xlsx`, mirrors Shorkie/Borzoi):**
- Head: last hidden state `[B,8192,1536]` → RC-average → `[B,8192,768]` → mean-pool
  per 16 bp bin → `[B,512,768]` → `Linear(768 → 4,201)` → `softplus`.
- Loss: **Poisson + Multinomial**, multinomial weighted **5×** the Poisson term.
  (≡ Borzoi/Shorkie `poisson_multinomial` with `total_weight=0.2`.)
- Targets: per-track coverage streamed on-the-fly from BigWigs via `pyBigWig`
  (`bw.stats(..., nBins=512)`), `log1p`-normalised; val/test pre-cached.
- Backbone: Phase-2 MLM LoRA **merged into** the backbone, then a **fresh** LoRA
  + head trained here (same LoRA config as Phase 2).

**Per-session run order (fresh Colab A100):** run cells top-to-bottom 1 → 8.
**NEVER re-run Cell 5** in the same session (PEFT-hook re-wrap) — restart runtime instead.

**Prerequisites (gated):**
1. Phase-2 MLM adapter saved to Drive (`checkpoints/phase2/...`). If absent, set
   `USE_MLM_ADAPTER = False` to scaffold-train from the base backbone.
2. Drive mounted + GCS/Drive auth (the BigWigs live on Drive).

> ⚠️ This notebook is a **scaffold** — authored against the existing notebooks'
> conventions and the WBS, but not yet executed end-to-end. Confirm the `SUP_SEQ_DIR`
> / `MANIFEST` / `MLM_ADAPTER` paths against notebook 01's `OUT_SUP_SEQ` and
> notebook 02's `CHECKPOINT_DIR` before a full run.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install   (IDENTICAL 4-step order to notebooks 00/02 — do NOT collapse)
#
# torch 2.3.1+cu121 → mamba-ssm 2.2.2 (built vs transformers==4.40.0)
# → causal-conv1d 1.4.0 (separate Mamba2 CUDA kernel; omitting it → runtime
#   AttributeError: 'NoneType'...causal_conv1d_fwd) → upgrade transformers+peft.
# peft==0.14.0 required for eva_config in PlantCAD2 adapter_config.json.
# ─────────────────────────────────────────────────────────────────────────────

# Step 1 — PyTorch pinned to cu121 (overrides Colab default cu128)
!pip3 install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121 -q

# Step 2 — mamba-ssm prebuilt wheel; transformers==4.40.0 is a temporary pin
!pip3 install mamba-ssm==2.2.2 transformers==4.40.0 -q

# Step 2a — causal-conv1d: separate Mamba2 CUDA kernel (MUST follow mamba-ssm)
!pip3 install causal-conv1d==1.4.0 -q

# Step 3 — Upgrade transformers + peft after mamba-ssm/causal-conv1d are built
!pip3 install transformers==4.46.3 peft==0.14.0 -q

# Step 4 — Remaining deps
!pip3 install biopython pyfaidx pyBigWig datasets \
    scikit-learn scipy pandas numpy tqdm huggingface_hub -q

import subprocess
expected = {'torch':'2.3.1+cu121','mamba_ssm':'2.2.2','causal_conv1d':'1.4.0',
            'transformers':'4.46.3','peft':'0.14.0'}
print('Package versions:')
all_ok = True
for pkg, exp in expected.items():
    r = subprocess.run(['pip','show',pkg.replace('_','-')],capture_output=True,text=True)
    ver = next((l.split(': ')[1] for l in r.stdout.splitlines() if l.startswith('Version')),'NOT FOUND')
    ok = '✅' if exp in ver else '❌'
    if '❌' in ok: all_ok = False
    print(f'  {ok} {pkg}: {ver} (expected: {exp})')
print(f"\n{'✅ All versions correct' if all_ok else '❌ Version mismatch — re-run this cell'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Mount Drive, paths, constants
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE         = Path('/content/drive/MyDrive/yeastcaduceus')
# R64 supervised SEQUENCE windows from notebook 01 (save_to_disk; fields:
# chrom/start/end/split/sequence/is_repeat). Confirm subpath = notebook 01 OUT_SUP_SEQ.
SUP_SEQ_DIR  = BASE / 'data/supervised_seq'           # TODO: verify vs notebook 01
# BigWigs live under the shorkie backup path (see notebook 01 Cell 7).
BIGWIG_DIR   = Path('/content/drive/MyDrive/shorkie/data/backup/supervised/bigwigs')
MANIFEST     = BASE / 'data/manifests/bigwig_manifest.json'   # from notebook 01 Cell 7
COVERAGE_CACHE = BASE / 'data/coverage_cache'         # pre-extracted val/test targets
CHECKPOINT_DIR = BASE / 'checkpoints/phase3'
RESULTS_DIR    = BASE / 'results/phase3'
# Phase-2 MLM adapter (notebook 02 CHECKPOINT_DIR / final adapter). If not yet
# saved (Phase 2 still running), set USE_MLM_ADAPTER=False to train from base.
MLM_ADAPTER  = BASE / 'checkpoints/phase2/final_adapter'      # TODO: verify
USE_MLM_ADAPTER = MLM_ADAPTER.exists()

MODEL_ID = 'kuleshov-group/PlantCAD2-Small-l24-d0768'
for d in (COVERAGE_CACHE, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Window / binning ──────────────────────────────────────────────────────────
WINDOW_SIZE = 8192
BIN_SIZE    = 16
N_BINS      = WINDOW_SIZE // BIN_SIZE      # 512

# ── Training (same OOM-bound regime as Phase 2: no grad-checkpointing in Caduceus)
PHYSICAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS    = 64                   # effective batch = 128
LEARNING_RATE       = 1e-4                 # WBS sweep: 1e-4 / 2e-4 / 5e-4
NUM_EPOCHS          = 3
WARMUP_STEPS        = 50
LOG_STEPS           = 50

# ── Loss ──────────────────────────────────────────────────────────────────────
MULTINOMIAL_WEIGHT  = 5.0   # multinomial : poisson = 5 : 1  ≡ Borzoi total_weight=0.2

# ── LoRA (identical to Phase 2 / PlantCAD2 paper Methods) ─────────────────────
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 32, 0.1
LORA_TARGET_MODULES = ['out_proj', 'x_proj', 'in_proj']

print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Effective batch     : {PHYSICAL_BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {PHYSICAL_BATCH_SIZE*GRAD_ACCUM_STEPS}')
print(f'Use Phase-2 adapter : {USE_MLM_ADAPTER}  ({MLM_ADAPTER if USE_MLM_ADAPTER else "training from BASE backbone"})')
print('\n✅ Cell 2 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Tokenizer, backbone (+ optional Phase-2 adapter merge), fresh LoRA
#
# Same tokenizer/vocab traps as Phase 2:
#   NUCLEOTIDE_IDS from tokenizer(nuc, add_special_tokens=False) (A=3,C=4,G=5,T=6).
#   Backbone is loaded bf16. If the Phase-2 MLM LoRA exists, it is MERGED into the
#   backbone (merge_and_unload) so the yeast domain adaptation is baked in, then a
#   FRESH supervised LoRA is attached for this task.
# ─────────────────────────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
_nuc = {}
for n in ['A','C','G','T']:
    ids = tokenizer(n, add_special_tokens=False)['input_ids']
    assert ids, f'empty ids for {n}'
    _nuc[n] = ids[0]
NUCLEOTIDE_IDS = list(_nuc.values())
print(f'  Nucleotide IDs: {_nuc}')

print('\nLoading PlantCAD2-Small backbone (bf16)...')
backbone = AutoModelForMaskedLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16)

if USE_MLM_ADAPTER:
    print(f'Merging Phase-2 MLM adapter from {MLM_ADAPTER} ...')
    backbone = PeftModel.from_pretrained(backbone, str(MLM_ADAPTER))
    backbone = backbone.merge_and_unload()      # bake yeast adaptation into weights
    print('  ✅ MLM adapter merged into backbone')
else:
    print('  ⚠️  No Phase-2 adapter — training supervised head on the BASE backbone')

# Fresh supervised LoRA
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES, bias='none',
    task_type=TaskType.FEATURE_EXTRACTION)
peft_model = get_peft_model(backbone, lora_config)

# VOCAB_SIZE sanity (lm_head=8, not tokenizer.vocab_size=7) — head ignores lm_head
VOCAB_SIZE = peft_model.base_model.model.lm_head.weight.shape[0]
print(f'  VOCAB_SIZE (lm_head): {VOCAB_SIZE}')

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in peft_model.parameters())
print(f'  LoRA trainable: {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)')
print('\n✅ Cell 3 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Poisson–Multinomial loss  (Borzoi/Shorkie)
#
# For each track, decompose coverage into:
#   • total  → Poisson(sum over bins)
#   • shape  → Multinomial(profile across bins)
# loss = poisson + MULTINOMIAL_WEIGHT * multinomial   (5:1 == Borzoi total_weight=0.2)
# Inputs are NON-negative (softplus pred; log1p targets ≥ 0). Shapes [B, N_BINS, T].
# ─────────────────────────────────────────────────────────────────────────────
import torch, torch.nn.functional as F

def poisson_multinomial(y_pred, y_true, multinomial_weight=MULTINOMIAL_WEIGHT, eps=1e-7):
    # y_pred, y_true: [B, N_BINS, T]   (bins on dim=1)
    y_pred = y_pred.float() + eps
    y_true = y_true.float() + eps
    s_pred = y_pred.sum(dim=1, keepdim=True)          # [B,1,T] total
    s_true = y_true.sum(dim=1, keepdim=True)
    # Poisson on the total (per track), mean over batch+tracks
    poisson = (s_pred - s_true * torch.log(s_pred)).squeeze(1)    # [B,T]
    # Multinomial on the profile: -sum_bins true * log(pred_prob)
    p_pred  = y_pred / s_pred                          # [B,N_BINS,T]
    multinom = -(y_true * torch.log(p_pred)).sum(dim=1)          # [B,T]
    return (poisson + multinomial_weight * multinom).mean()

# ── unit sanity: identical pred/true → near-minimal; random → larger ──────────
_t = torch.rand(2, N_BINS, 8) * 5
_loss_same = poisson_multinomial(_t, _t).item()
_loss_rand = poisson_multinomial(torch.rand(2, N_BINS, 8) * 5, _t).item()
print(f'loss(true,true) = {_loss_same:.3f}   loss(rand,true) = {_loss_rand:.3f}')
assert _loss_rand > _loss_same, 'multinomial term not penalising profile mismatch'
print('\n✅ Cell 4 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Coverage head + model wrapper
#
# CaduceusForMaskedLM accepts input_ids + output_hidden_states (NOT attention_mask).
# We read hidden_states[-1] through the PEFT forward hooks (so LoRA routes), then:
#   RC-average (1536 → 768) → mean-pool per 16bp bin (8192 → 512) → Linear → softplus.
# Head kept in fp32 for stable Poisson/Multinomial; backbone stays bf16.
#
# !! NEVER RE-RUN THIS CELL in the same session — re-wrapping the PEFT hooks twice
#    is the same recursion trap as Phase 2 Cell 4. Restart runtime to re-run. !!
# ─────────────────────────────────────────────────────────────────────────────
import json, torch, torch.nn as nn, torch.nn.functional as F

with open(MANIFEST) as f:
    NUM_TRACKS = json.load(f)['n_tracks']
print(f'NUM_TRACKS (from manifest): {NUM_TRACKS}')

class CoverageModel(nn.Module):
    """Wraps the PEFT backbone with a binned coverage-regression head."""
    def __init__(self, peft_model, num_tracks, hidden=768, bin_size=BIN_SIZE):
        super().__init__()
        self.peft_model = peft_model
        self.bin_size   = bin_size
        self.head = nn.Linear(hidden, num_tracks).float()

    def _embed(self, input_ids):
        # route through PEFT hooks; request hidden states (Caduceus supports this)
        with self.peft_model._enable_peft_forward_hooks():
            out = self.peft_model.base_model(input_ids=input_ids,
                                             output_hidden_states=True)
        hs = out.hidden_states[-1]                     # [B, L, 1536]
        h = hs.shape[-1] // 2
        emb = (hs[..., :h] + hs[..., h:].flip(-1)) / 2 # RC-average → [B, L, 768]
        B, L, H = emb.shape
        emb = emb.view(B, L // self.bin_size, self.bin_size, H).mean(2)  # [B,512,768]
        return emb.float()

    def forward(self, input_ids=None, coverage=None, **kw):
        pred = F.softplus(self.head(self._embed(input_ids)))   # [B,512,T] ≥ 0
        loss = poisson_multinomial(pred, coverage) if coverage is not None else None
        return {'loss': loss, 'pred': pred}

model = CoverageModel(peft_model, NUM_TRACKS).to('cuda:0')

# sanity forward: random window + random targets → finite loss, correct shape
_ids = torch.randint(min(NUCLEOTIDE_IDS), max(NUCLEOTIDE_IDS)+1, (2, WINDOW_SIZE), device='cuda')
_cov = torch.rand(2, N_BINS, NUM_TRACKS, device='cuda')
_o = model(input_ids=_ids, coverage=_cov)
assert _o['pred'].shape == (2, N_BINS, NUM_TRACKS), _o['pred'].shape
assert torch.isfinite(_o['loss']), 'non-finite loss'
print(f"  pred {tuple(_o['pred'].shape)}  loss {_o['loss'].item():.3f}")
del _ids, _cov, _o
print('\n✅ Cell 5 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — BigWig streaming dataset + collator
#
# Each item: R64 window (chrom,start) → tokenized input_ids [8192] + coverage
# targets [N_BINS, NUM_TRACKS]. Coverage read on-the-fly via pyBigWig
# bw.stats(chrom,start,end,nBins=512,type='mean'); None bins → 0.0; log1p per track.
# (WBS specifies per-track MEDIAN impute — compute medians in the cache step and
#  swap in below; 0.0 is the simple scaffold default.)
#
# BigWig handles are opened lazily PER WORKER (pyBigWig handles are not fork-safe).
# Val/test should be pre-extracted to COVERAGE_CACHE; train streams.
# ─────────────────────────────────────────────────────────────────────────────
import json, numpy as np, torch, pyBigWig
from pathlib import Path
from torch.utils.data import Dataset
from datasets import load_from_disk

with open(MANIFEST) as f:
    _man = json.load(f)
BW_PATHS = [str(BIGWIG_DIR / t['filename']) for t in _man['tracks']]
assert len(BW_PATHS) == NUM_TRACKS

class BigWigCoverageDataset(Dataset):
    def __init__(self, split):
        self.windows = load_from_disk(str(SUP_SEQ_DIR / split))
        self.bws = None                      # opened per worker

    def _ensure_open(self):
        if self.bws is None:
            self.bws = [pyBigWig.open(p) for p in BW_PATHS]

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        self._ensure_open()
        w = self.windows[i]
        ids = tokenizer(w['sequence'], return_tensors='pt')['input_ids'][0]  # [8192]
        cov = np.zeros((N_BINS, NUM_TRACKS), dtype=np.float32)
        for t, bw in enumerate(self.bws):
            vals = bw.stats(w['chrom'], int(w['start']), int(w['end']),
                            nBins=N_BINS, type='mean')
            cov[:, t] = [0.0 if v is None else v for v in vals]
        cov = np.log1p(cov)                  # Shorkie convention
        return {'input_ids': ids, 'coverage': torch.from_numpy(cov)}

def collate(batch):
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'coverage':  torch.stack([b['coverage']  for b in batch]),
    }

# quick smoke (1 window) — comment out if Drive/BigWigs not yet mounted
# _ds = BigWigCoverageDataset('val'); _b = collate([_ds[0]])
# print('input_ids', _b['input_ids'].shape, 'coverage', _b['coverage'].shape)
print('✅ Cell 6 defined (BigWigCoverageDataset, collate)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Train (HF Trainer)
#
# bf16, no grad-checkpointing (Caduceus lacks it), batch 2 × accum 64.
# Trainer routes model(**inputs) → CoverageModel.forward → dict with 'loss'.
# ─────────────────────────────────────────────────────────────────────────────
from transformers import Trainer, TrainingArguments

class CoverageTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        out = model(input_ids=inputs['input_ids'], coverage=inputs['coverage'])
        return (out['loss'], out) if return_outputs else out['loss']

train_ds = BigWigCoverageDataset('train')
val_ds   = BigWigCoverageDataset('val')

args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=PHYSICAL_BATCH_SIZE,
    per_device_eval_batch_size=PHYSICAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOG_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    bf16=True,
    gradient_checkpointing=False,            # not implemented in Caduceus
    dataloader_num_workers=2,
    remove_unused_columns=False,             # keep 'coverage'
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

trainer = CoverageTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collate,
)
trainer.train()

# save LoRA adapter + head separately
peft_model.save_pretrained(str(CHECKPOINT_DIR / 'final_adapter'))
torch.save(model.head.state_dict(), CHECKPOINT_DIR / 'coverage_head.pt')
print('✅ Cell 7 — training done, adapter + head saved')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Evaluate: bin-level Pearson R + per-track Spearman  (test chroms)
#
# Baselines (Shorkie): bin-level median R ≈ 0.78, gene-level mean R ≈ 0.88.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np, torch
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import DataLoader

model.eval()
test_ds = BigWigCoverageDataset('test')
loader  = DataLoader(test_ds, batch_size=PHYSICAL_BATCH_SIZE, collate_fn=collate)

preds, trues = [], []
with torch.no_grad():
    for b in loader:
        out = model(input_ids=b['input_ids'].to('cuda:0'))
        preds.append(out['pred'].float().cpu().numpy())
        trues.append(b['coverage'].numpy())
P = np.concatenate(preds)   # [N,512,T]
T = np.concatenate(trues)

# bin-level Pearson per window (flatten bins×tracks), then median across windows
bin_r = [pearsonr(P[i].ravel(), T[i].ravel())[0] for i in range(len(P))]
# per-track Spearman across all (window,bin) positions
flatP, flatT = P.reshape(-1, P.shape[-1]), T.reshape(-1, T.shape[-1])
track_rho = [spearmanr(flatP[:, t], flatT[:, t]).correlation for t in range(P.shape[-1])]

import json
metrics = {
    'bin_pearson_median': float(np.nanmedian(bin_r)),
    'bin_pearson_mean':   float(np.nanmean(bin_r)),
    'track_spearman_median': float(np.nanmedian(track_rho)),
    'n_test_windows': int(len(P)),
}
print(metrics)
print(f"  → Shorkie bin-level median R baseline ≈ 0.78")
(RESULTS_DIR / 'phase3_metrics.json').write_text(json.dumps(metrics, indent=2))
print('✅ Cell 8 — metrics saved')